# Merge, Join, and Reshape

Real-world data analysis almost always requires combining data from multiple sources. This notebook covers the essential operations for merging DataFrames and reshaping data between wide and long formats. These skills are critical for data preparation.

## Learning Objectives

- Understand different merge types (inner, left, right, outer)
- Use `pd.merge()` for column-based joins and `.join()` for index-based joins
- Concatenate DataFrames with `pd.concat()`
- Reshape data with `pivot_table()` and `melt()`
- Use `stack()` and `unstack()` for MultiIndex manipulation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Load data
df = pd.read_csv('../../data/titanic.csv')
print(f"Shape: {df.shape}")
df.head()

## Sample Data for Merge Examples

In [ ]:
# Create sample tables for demonstration
# Passengers table
passengers = pd.DataFrame({
    'passenger_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'cabin_id': ['A1', 'B2', 'A1', 'C3', 'D4']
})

# Cabins table (note: D4 is not in cabins, F5 is not in passengers)
cabins = pd.DataFrame({
    'cabin_id': ['A1', 'B2', 'C3', 'F5'],
    'deck': ['A', 'B', 'C', 'F'],
    'capacity': [2, 1, 1, 3]
})

print("Passengers:")
print(passengers)
print("\nCabins:")
print(cabins)

## pd.merge() - Column-Based Joins

The `merge()` function joins DataFrames based on column values.

In [ ]:
# Visualize join types
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

def draw_venn(ax, title, left_only, both, right_only, highlight):
    # Draw circles
    circle1 = plt.Circle((0.35, 0.5), 0.3, fill=False, color='blue', linewidth=2)
    circle2 = plt.Circle((0.65, 0.5), 0.3, fill=False, color='red', linewidth=2)
    ax.add_patch(circle1)
    ax.add_patch(circle2)
    
    # Highlight regions
    colors = {'left': '#3498db44', 'both': '#9b59b644', 'right': '#e74c3c44'}
    if 'left' in highlight:
        wedge1 = patches.Wedge((0.35, 0.5), 0.3, 90, 270, color='#3498db88')
        ax.add_patch(wedge1)
    if 'both' in highlight:
        # Draw intersection
        theta = np.linspace(0, 2*np.pi, 100)
        ax.fill_between([0.35, 0.65], [0.2, 0.2], [0.8, 0.8], color='#9b59b688', alpha=0.5)
    if 'right' in highlight:
        wedge2 = patches.Wedge((0.65, 0.5), 0.3, 270, 90, color='#e74c3c88')
        ax.add_patch(wedge2)
    
    # Labels
    ax.text(0.2, 0.5, f'{left_only}', ha='center', va='center', fontsize=14)
    ax.text(0.5, 0.5, f'{both}', ha='center', va='center', fontsize=14)
    ax.text(0.8, 0.5, f'{right_only}', ha='center', va='center', fontsize=14)
    ax.text(0.35, 0.9, 'Left', ha='center', fontsize=12, color='blue')
    ax.text(0.65, 0.9, 'Right', ha='center', fontsize=12, color='red')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')

draw_venn(axes[0, 0], 'INNER JOIN\n(only matching rows)', 'Eve', 'Alice\nBob\nCharlie\nDiana', 'F5', ['both'])
draw_venn(axes[0, 1], 'LEFT JOIN\n(all from left + matches)', 'Eve', 'Alice\nBob\nCharlie\nDiana', '-', ['left', 'both'])
draw_venn(axes[1, 0], 'RIGHT JOIN\n(all from right + matches)', '-', 'Alice\nBob\nCharlie\nDiana', 'F5', ['both', 'right'])
draw_venn(axes[1, 1], 'OUTER JOIN\n(everything)', 'Eve', 'Alice\nBob\nCharlie\nDiana', 'F5', ['left', 'both', 'right'])

plt.tight_layout()
plt.show()

In [ ]:
# Inner join - only matching rows
inner = pd.merge(passengers, cabins, on='cabin_id', how='inner')
print("INNER JOIN:")
inner

In [ ]:
# Left join - all from left, match from right
left = pd.merge(passengers, cabins, on='cabin_id', how='left')
print("LEFT JOIN:")
left

In [ ]:
# Right join - all from right, match from left
right = pd.merge(passengers, cabins, on='cabin_id', how='right')
print("RIGHT JOIN:")
right

In [ ]:
# Outer join - all rows from both
outer = pd.merge(passengers, cabins, on='cabin_id', how='outer')
print("OUTER JOIN:")
outer

### Merge with Different Column Names

In [ ]:
# When join columns have different names
cabins_renamed = cabins.rename(columns={'cabin_id': 'cabin_number'})
print("Cabins with renamed column:")
print(cabins_renamed)

# Use left_on and right_on
merged = pd.merge(
    passengers, cabins_renamed,
    left_on='cabin_id', right_on='cabin_number',
    how='left'
)
print("\nMerged with different column names:")
merged

> **Gotcha: Duplicate columns**
> 
> When both DataFrames have columns with the same name (other than join keys), Pandas adds `_x` and `_y` suffixes. Use the `suffixes=` parameter to customize them.

## .join() - Index-Based Joins

In [ ]:
# Set index for join
passengers_idx = passengers.set_index('cabin_id')
cabins_idx = cabins.set_index('cabin_id')

print("Passengers (indexed):")
print(passengers_idx)
print("\nCabins (indexed):")
print(cabins_idx)

In [ ]:
# Index-based join
joined = passengers_idx.join(cabins_idx, how='left')
print("Index-based join:")
joined

## pd.concat() - Stacking DataFrames

In [ ]:
# Create sample DataFrames
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df2 = pd.DataFrame({'A': [5, 6], 'B': [7, 8]})
df3 = pd.DataFrame({'C': [9, 10], 'D': [11, 12]})

print("df1:")
print(df1)
print("\ndf2:")
print(df2)

In [ ]:
# Vertical concatenation (stack rows)
vertical = pd.concat([df1, df2], axis=0, ignore_index=True)
print("Vertical concat (axis=0):")
vertical

In [ ]:
# Horizontal concatenation (stack columns)
horizontal = pd.concat([df1, df3], axis=1)
print("Horizontal concat (axis=1):")
horizontal

In [ ]:
# Keys for hierarchical index
keyed = pd.concat([df1, df2], keys=['first', 'second'])
print("Concat with keys:")
keyed

## pivot_table() - Reshaping to Wide Format

In [ ]:
# Pivot table: survival rate by class and sex
pivot = df.pivot_table(
    values='Survived',
    index='Pclass',
    columns='Sex',
    aggfunc='mean'
)
print("Pivot table - Survival rate:")
pivot.round(3)

In [ ]:
# Multiple values and aggregations
pivot_multi = df.pivot_table(
    values=['Survived', 'Fare'],
    index='Pclass',
    columns='Sex',
    aggfunc={'Survived': 'mean', 'Fare': ['mean', 'median']}
)
print("Multi-value pivot:")
pivot_multi.round(2)

In [ ]:
# With margins (totals)
pivot_margins = df.pivot_table(
    values='Survived',
    index='Pclass',
    columns='Sex',
    aggfunc='mean',
    margins=True,
    margins_name='Total'
)
print("Pivot with margins:")
pivot_margins.round(3)

## melt() - Reshaping to Long Format

The `melt()` function is the inverse of pivot - it converts wide data to long format. This is essential for "tidy data" principles.

In [ ]:
# Wide format data
wide_data = pd.DataFrame({
    'name': ['Alice', 'Bob'],
    'math_score': [90, 85],
    'english_score': [88, 92],
    'science_score': [95, 78]
})
print("Wide format:")
wide_data

In [ ]:
# Melt to long format
long_data = pd.melt(
    wide_data,
    id_vars=['name'],
    value_vars=['math_score', 'english_score', 'science_score'],
    var_name='subject',
    value_name='score'
)
print("Long format (melted):")
long_data

## stack() and unstack() for MultiIndex

In [ ]:
# Create MultiIndex DataFrame
multi_df = df.groupby(['Pclass', 'Sex'])['Survived'].mean().round(3)
print("MultiIndex Series:")
print(multi_df)

In [ ]:
# Unstack - move inner index to columns
unstacked = multi_df.unstack()
print("Unstacked:")
unstacked

In [ ]:
# Stack - move columns to index
stacked = unstacked.stack()
print("Stacked back:")
stacked

## Exercises

### Exercise 1: Merge Practice

Create two DataFrames:
1. `employees`: id, name, dept_id
2. `departments`: dept_id, dept_name, budget

Include some employees without departments and some departments without employees. Then perform all four types of joins.

In [ ]:
# Create DataFrames and perform merges
# YOUR CODE HERE

### Exercise 2: Pivot Table

Using the Titanic data, create a pivot table showing:
- Rows: Embarked (port)
- Columns: Pclass
- Values: Count of passengers AND average fare

In [ ]:
# Create the pivot table
# YOUR CODE HERE

### Exercise 3: Melt Practice

The Titanic dataset has columns SibSp (siblings/spouses) and Parch (parents/children). Create a melted version that has:
- id_vars: PassengerId, Survived
- A 'relation_type' column (SibSp or Parch)
- A 'count' column with the values

In [ ]:
# Melt family relations columns
# YOUR CODE HERE

### Exercise 4: Concatenation

Split the Titanic dataset into three DataFrames by Pclass, then concatenate them back together with a hierarchical index showing which class each row came from.

In [ ]:
# Split and concatenate with keys
# YOUR CODE HERE